# ⚽ World Cup 2026 — AI Predictor
## Day 2: Model Training, Validation & Group Stage Predictions

**What we build today:**

| Step | Description | Why it matters |
|------|-------------|----------------|
| 1 | Load Day 1 outputs | Everything flows from the features we built |
| 2 | Understand the problem | 3-class classification, class imbalance, the draw problem |
| 3 | Baseline model | FIFA ranking as naive predictor — our floor |
| 4 | Logistic Regression | Linear model with Elo features |
| 5 | LightGBM | Gradient boosting — our main model |
| 6 | Leave-one-tournament-out | Real validation: Qatar 2022 as blind test |
| 7 | Error analysis | Why did we miss Morocco? What does that teach us? |
| 8 | WC 2026 team features | Build the input for predictions |
| 9 | Group stage predictions | Full bracket with probabilities |

---

### 🧠 Before you run: understand the problem

We're predicting football match outcomes. That sounds simple. It's not.

**Why football is hard to predict:**
- 3 possible outcomes: home win / draw / away win (not binary!)
- Draws happen ~25% of the time in group stage, ~0% in knockouts
- Upsets are real and frequent — the 'weaker' team wins ~30% of the time
- Small sample sizes: only 64 matches per World Cup
- The sport has high variance — one moment (red card, penalty) changes everything

**What this means for our model:**
- Accuracy alone is misleading (a model that never predicts draws still gets ~55% accuracy)
- We need F1 score per class — especially for the draw class
- Probabilistic outputs (predict_proba) are more useful than hard predictions
- Our goal: quantify uncertainty better than a ranking table does

---

## 0. Setup

In [1]:
# ── Install ────────────────────────────────────────────────────────────────
!pip install -q lightgbm scikit-learn plotly

# ── Imports ────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, log_loss, f1_score
)
import lightgbm as lgb

pd.set_option('display.float_format', '{:.3f}'.format)
print('✅ Setup complete')

✅ Setup complete


In [2]:
# ── Mount Drive & set paths ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = '/content/drive/MyDrive/World Cup Data/Data/'  # ← adjust if needed

def load(filename):
    path = DATA_PATH + filename
    try:
        return pd.read_csv(path, encoding='utf-8')
    except:
        return pd.read_csv(path, encoding='latin-1')

print('✅ Drive mounted')

Mounted at /content/drive
✅ Drive mounted


## 1. Load Day 1 Outputs

We rebuild the feature pipeline here from scratch — this makes the notebook self-contained and reproducible.
In production, you'd load the saved CSVs from Day 1. Here we rebuild so you can see every step.

In [3]:
# ── Load raw data ──────────────────────────────────────────────────────────
matches   = load('matches_1930_2022.csv')
ranking   = load('fifa_ranking_2022-10-06.csv')
p_stats   = load('player_stats.csv')
p_gca     = load('player_gca.csv')
p_defense = load('player_defense.csv')
p_shoot   = load('player_shooting.csv')
team_data = load('team_data.csv')

print(f'Matches loaded:  {matches.shape}')
print(f'Ranking loaded:  {ranking.shape}')
print(f'Players loaded:  {p_stats.shape}')

Matches loaded:  (964, 44)
Ranking loaded:  (211, 7)
Players loaded:  (680, 31)


In [4]:
# ── Rebuild match outcomes ─────────────────────────────────────────────────
def get_result(row):
    """Outcome from home team perspective."""
    if row['home_score'] > row['away_score']:   return 'home_win'
    elif row['home_score'] < row['away_score']: return 'away_win'
    else:                                        return 'draw'

matches['result']      = matches.apply(get_result, axis=1)
matches['total_goals'] = matches['home_score'] + matches['away_score']

# Encode target: 0=home_win, 1=draw, 2=away_win
TARGET_MAP = {'home_win': 0, 'draw': 1, 'away_win': 2}
REV_MAP    = {0: 'home_win', 1: 'draw', 2: 'away_win'}
matches['target'] = matches['result'].map(TARGET_MAP)

# Class distribution — important to understand before modeling
print('Overall result distribution:')
dist = matches['result'].value_counts()
for r, n in dist.items():
    pct = n/len(matches)
    bar = '█' * int(pct * 40)
    print(f'  {r:<12} {n:>4} ({pct:.1%})  {bar}')

print()
print('By stage:')
knockout = ['Round of 16','Quarter-finals','Semi-finals','Final','Third-place match']
for stage, label in [('Group stage', 'group'), (knockout, 'knockout')]:
    if isinstance(stage, list):
        sub = matches[matches['Round'].isin(stage)]
    else:
        sub = matches[matches['Round'] == stage]
    draw_pct = (sub['result'] == 'draw').mean()
    print(f'  {label}: draw rate = {draw_pct:.1%}')

print()
print('💡 Key insight: draw rate drops dramatically in knockouts.')
print('   This means is_knockout is a critical feature for the model.')

Overall result distribution:
  home_win      532 (55.2%)  ██████████████████████
  away_win      218 (22.6%)  █████████
  draw          214 (22.2%)  ████████

By stage:
  group: draw rate = 23.7%
  knockout: draw rate = 15.9%

💡 Key insight: draw rate drops dramatically in knockouts.
   This means is_knockout is a critical feature for the model.


In [5]:
# ── Rebuild Elo ratings ───────────────────────────────────────────────────
# (Full explanation of Elo in Day 1 notebook)
# Here we run it fast — same logic, more concise

def compute_elo(matches_df, k=32, initial=1500):
    elo = {}
    history = []
    for _, row in matches_df.sort_values('Year').iterrows():
        h, a, yr = row['home_team'], row['away_team'], row['Year']
        if h not in elo: elo[h] = initial
        if a not in elo: elo[a] = initial
        rh, ra = elo[h], elo[a]
        exp_h = 1 / (1 + 10 ** ((ra - rh) / 400))
        act_h = 1.0 if row['result']=='home_win' else (0.0 if row['result']=='away_win' else 0.5)
        new_h = rh + k * (act_h - exp_h)
        new_a = ra + k * ((1-act_h) - (1-exp_h))
        history += [
            {'year': yr, 'team': h, 'elo_after': new_h},
            {'year': yr, 'team': a, 'elo_after': new_a}
        ]
        elo[h], elo[a] = new_h, new_a
    return elo, pd.DataFrame(history)

elo_final, elo_hist = compute_elo(matches)

# Build Elo-at-entry feature: what was each team's Elo BEFORE each WC?
# This is the correct way — prevents data leakage
elo_entries = []
for year in sorted(matches['Year'].unique()):
    teams = set(matches[matches['Year']==year]['home_team'].tolist() +
                matches[matches['Year']==year]['away_team'].tolist())
    for team in teams:
        prior = elo_hist[(elo_hist['team']==team) & (elo_hist['year']<year)]
        elo_entries.append({
            'team': team, 'year': year,
            'elo_entry': prior.iloc[-1]['elo_after'] if len(prior) > 0 else 1500
        })
elo_entry_df = pd.DataFrame(elo_entries)

print('✅ Elo computed for', len(elo_final), 'teams')
print('Top 5 by final Elo (post Qatar 2022):')
top5 = sorted(elo_final.items(), key=lambda x: -x[1])[:5]
for team, rating in top5:
    print(f'  {team:<20} {rating:.0f}')

✅ Elo computed for 86 teams
Top 5 by final Elo (post Qatar 2022):
  Netherlands          1704
  Brazil               1693
  France               1693
  Germany              1667
  Argentina            1656


In [6]:
# ── Rebuild historical win rates ───────────────────────────────────────────
# Win rate = wins / total matches played, across all World Cups
# This captures a different thing than Elo:
#   Elo = how strong you are relative to opponents you faced
#   Win rate = raw proportion of games won (simpler, but still informative)

home_w  = matches[matches['result']=='home_win'].groupby('home_team').size()
away_w  = matches[matches['result']=='away_win'].groupby('away_team').size()
home_p  = matches.groupby('home_team').size()
away_p  = matches.groupby('away_team').size()
total_w = home_w.add(away_w, fill_value=0)
total_p = home_p.add(away_p, fill_value=0)
win_rate = (total_w / total_p).fillna(0.33)

print('Win rates (top 10 teams, min 10 matches):')
wr_df = pd.DataFrame({'win_rate': win_rate, 'total_matches': total_p})
print(wr_df[wr_df['total_matches'] >= 10].sort_values('win_rate', ascending=False)
      .head(10).round(3).to_string())

Win rates (top 10 teams, min 10 matches):
              win_rate  total_matches
Brazil           0.667        114.000
Germany          0.661         56.000
West Germany     0.554         56.000
Netherlands      0.545         55.000
Italy            0.542         83.000
France           0.534         73.000
Argentina        0.534         88.000
Türkiye          0.500         10.000
Portugal         0.486         35.000
Soviet Union     0.484         31.000


## 2. Build the ML Dataset

Each row = one match.  
Features = what we know BEFORE the match is played.  
Target = what actually happened.

**The most important rule in ML: no data leakage.**  
We only use information that would have been available before each match was played.

In [7]:
# ── Build match-level feature table ───────────────────────────────────────
mf = matches.copy()

# Add Elo for home team
mf = mf.merge(
    elo_entry_df.rename(columns={'team':'home_team','elo_entry':'elo_home','year':'Year'}),
    on=['home_team','Year'], how='left'
)
# Add Elo for away team
mf = mf.merge(
    elo_entry_df.rename(columns={'team':'away_team','elo_entry':'elo_away','year':'Year'}),
    on=['away_team','Year'], how='left'
)

# Difference features — the model learns from gaps, not absolutes
mf['elo_diff']    = mf['elo_home'] - mf['elo_away']   # + = home team stronger
mf['wr_home']     = mf['home_team'].map(win_rate).fillna(0.33)
mf['wr_away']     = mf['away_team'].map(win_rate).fillna(0.33)
mf['wr_diff']     = mf['wr_home'] - mf['wr_away']     # + = home team historically better
mf['is_knockout'] = mf['Round'].isin([
    'Round of 16','Quarter-finals','Semi-finals','Final','Third-place match'
]).astype(int)

# Add FIFA ranking
rank_map = ranking.set_index('team')['rank'].to_dict()
mf['rank_home']  = mf['home_team'].map(rank_map)
mf['rank_away']  = mf['away_team'].map(rank_map)
# Note: lower rank number = BETTER team. So positive rank_diff = home is WORSE.
mf['rank_diff']  = mf['rank_home'] - mf['rank_away']

FEATURES = ['elo_diff', 'elo_home', 'elo_away', 'wr_diff', 'wr_home', 'wr_away', 'is_knockout']

# Train: everything before Qatar 2022
# Test:  Qatar 2022 only (leave-one-tournament-out)
train = mf[mf['Year'] < 2022].dropna(subset=FEATURES + ['target'])
test  = mf[mf['Year'] == 2022].dropna(subset=['elo_diff', 'target'])

X_train = train[FEATURES]
y_train = train['target']
X_test  = test[FEATURES].fillna(0)
y_test  = test['target']

print(f'Training set:  {len(train)} matches  ({train["Year"].min()}–{train["Year"].max()})')
print(f'Test set:      {len(test)} matches   (Qatar 2022 only — the model never saw these)')
print(f'Features used: {FEATURES}')
print()
print('Training class distribution:')
for k, v in y_train.value_counts().items():
    print(f'  {REV_MAP[k]:<12}: {v} ({v/len(y_train):.1%})')

Training set:  900 matches  (1930–2018)
Test set:      64 matches   (Qatar 2022 only — the model never saw these)
Features used: ['elo_diff', 'elo_home', 'elo_away', 'wr_diff', 'wr_home', 'wr_away', 'is_knockout']

Training class distribution:
  home_win    : 503 (55.9%)
  draw        : 199 (22.1%)
  away_win    : 198 (22.0%)


## 3. Baseline: FIFA Ranking

Before training any model, we need a **baseline** — the simplest possible predictor.
If our model can't beat this, it's useless.

Baseline rule: *the higher-ranked team (lower rank number) wins. If close, predict draw.*

In [8]:
# ── FIFA Ranking baseline ──────────────────────────────────────────────────
test_rank = test.dropna(subset=['rank_diff']).copy()

# Rule: if home team ranked 8+ places better → predict home win
#       if away team ranked 8+ places better → predict away win
#       otherwise → draw
# (The threshold 8 was chosen by trying a few values)
RANK_THRESHOLD = 8
test_rank['rank_pred'] = test_rank['rank_diff'].apply(
    lambda x: 2 if x > RANK_THRESHOLD else (0 if x < -RANK_THRESHOLD else 1)
)

rank_acc = accuracy_score(test_rank['target'], test_rank['rank_pred'])
rank_f1  = f1_score(test_rank['target'], test_rank['rank_pred'], average='weighted')

print('=== FIFA Ranking Baseline ===')
print(f'Accuracy:          {rank_acc:.1%}')
print(f'Weighted F1:       {rank_f1:.3f}')
print()
print(classification_report(
    test_rank['target'], test_rank['rank_pred'],
    target_names=['home_win','draw','away_win']
))
print('⚠️  This is our floor. Any model we build must beat this.')

=== FIFA Ranking Baseline ===
Accuracy:          51.7%
Weighted F1:       0.517

              precision    recall  f1-score   support

    home_win       0.70      0.68      0.69        28
        draw       0.25      0.23      0.24        13
    away_win       0.43      0.47      0.45        19

    accuracy                           0.52        60
   macro avg       0.46      0.46      0.46        60
weighted avg       0.52      0.52      0.52        60

⚠️  This is our floor. Any model we build must beat this.


## 4. Logistic Regression

First real model. Logistic Regression is a linear model — it learns a straight-line boundary between classes.

**Why start here?**
- Fast to train
- Interpretable — we can read the coefficients
- If it does well, the problem might be simpler than we think
- If it does poorly, we know we need non-linear models

In [9]:
# ── Logistic Regression ────────────────────────────────────────────────────
# Important: we MUST scale features for LR
# Elo values are ~1500, win rates are ~0.4 — very different scales
# Without scaling, the model gives too much weight to Elo just because the numbers are bigger

scaler  = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)   # fit on train only!
X_te_sc = scaler.transform(X_test)        # transform test with same scaler

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_tr_sc, y_train)
lr_pred  = lr.predict(X_te_sc)
lr_acc   = accuracy_score(y_test, lr_pred)
lr_f1    = f1_score(y_test, lr_pred, average='weighted')

print('=== Logistic Regression ===')
print(f'Accuracy:    {lr_acc:.1%}')
print(f'Weighted F1: {lr_f1:.3f}')
print()
print(classification_report(y_test, lr_pred, target_names=['home_win','draw','away_win']))

# Coefficients — what did the model learn?
print('What the model learned (coefficients):')
coef_df = pd.DataFrame(
    lr.coef_,
    columns=FEATURES,
    index=['home_win','draw','away_win']
)
print(coef_df.round(3).to_string())
print()
print('💡 Read this as: positive coefficient = feature pushes toward this outcome')
print('   e.g. high elo_diff (home team stronger) → positive for home_win, negative for away_win')

=== Logistic Regression ===
Accuracy:    46.9%
Weighted F1: 0.394

              precision    recall  f1-score   support

    home_win       0.51      0.79      0.62        29
        draw       0.00      0.00      0.00        15
    away_win       0.37      0.35      0.36        20

    accuracy                           0.47        64
   macro avg       0.29      0.38      0.33        64
weighted avg       0.35      0.47      0.39        64

What the model learned (coefficients):
          elo_diff  elo_home  elo_away  wr_diff  wr_home  wr_away  is_knockout
home_win    -0.090    -0.262    -0.177    0.309    0.449    0.032        0.107
draw         0.004     0.038     0.039    0.031   -0.078   -0.130       -0.126
away_win     0.085     0.225     0.138   -0.340   -0.370    0.098        0.019

💡 Read this as: positive coefficient = feature pushes toward this outcome
   e.g. high elo_diff (home team stronger) → positive for home_win, negative for away_win


## 5. LightGBM — Main Model

LightGBM is a **gradient boosting** algorithm. It builds many small decision trees, each one correcting the mistakes of the previous one.

**Why LightGBM over Random Forest or XGBoost?**
- Handles class imbalance better
- Faster on tabular data
- Provides feature importance natively
- Works well with small datasets (our 900-match training set is small for ML)

**Key hyperparameters we're setting:**
- `n_estimators=300` — number of trees
- `learning_rate=0.03` — how much each tree corrects the previous
- `num_leaves=15` — complexity of each tree (small = prevents overfitting on small dataset)
- `min_child_samples=5` — minimum matches per leaf (also prevents overfitting)

In [10]:
# ── LightGBM ───────────────────────────────────────────────────────────────
lgbm = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.03,
    num_leaves=15,
    min_child_samples=5,
    random_state=42,
    verbose=-1
)

lgbm.fit(X_train, y_train)
lgb_pred  = lgbm.predict(X_test)
lgb_proba = lgbm.predict_proba(X_test)   # probabilities for each outcome
lgb_acc   = accuracy_score(y_test, lgb_pred)
lgb_f1    = f1_score(y_test, lgb_pred, average='weighted')
lgb_ll    = log_loss(y_test, lgb_proba)   # log-loss: lower is better

print('=== LightGBM ===')
print(f'Accuracy:    {lgb_acc:.1%}')
print(f'Weighted F1: {lgb_f1:.3f}')
print(f'Log-Loss:    {lgb_ll:.3f}  ← measures probability calibration, lower = better')
print()
print(classification_report(y_test, lgb_pred, target_names=['home_win','draw','away_win']))
print()
print('💡 Log-loss matters more than accuracy here:')
print('   A model that says "70% home win" and is right is better than')
print('   one that says "100% home win" and is right — because the 70% model')
print('   is honest about uncertainty. That\'s what we want for predictions.')

=== LightGBM ===
Accuracy:    48.4%
Weighted F1: 0.478
Log-Loss:    1.204  ← measures probability calibration, lower = better

              precision    recall  f1-score   support

    home_win       0.65      0.69      0.67        29
        draw       0.31      0.27      0.29        15
    away_win       0.35      0.35      0.35        20

    accuracy                           0.48        64
   macro avg       0.43      0.44      0.43        64
weighted avg       0.47      0.48      0.48        64


💡 Log-loss matters more than accuracy here:
   A model that says "70% home win" and is right is better than
   one that says "100% home win" and is right — because the 70% model
   is honest about uncertainty. That's what we want for predictions.


In [11]:
# ── Feature Importance ────────────────────────────────────────────────────
# This tells us: which features did the model rely on most?

fi = dict(zip(FEATURES, lgbm.feature_importances_))

print('Feature Importance (LightGBM):')
print()
for feat, imp in sorted(fi.items(), key=lambda x: -x[1]):
    bar = '█' * int(imp / max(fi.values()) * 30)
    print(f'  {feat:<18} {imp:>5}  {bar}')

print()
print('What this means:')
print('  wr_diff  — Historical win rate gap between teams is the strongest signal')
print('  elo_diff — Elo gap is close second — captures recent performance trend')
print('  elo_away — The away team\'s absolute Elo matters more than home team\'s')
print('             (away teams must be truly strong to win at World Cups)')
print('  is_knockout — Least important. The model struggles to use stage info.')
print('               This is a weakness — draws are almost impossible in knockouts')
print('               but the model still sometimes predicts them there.')

Feature Importance (LightGBM):

  wr_diff             2642  ██████████████████████████████
  elo_diff            2478  ████████████████████████████
  elo_away            2266  █████████████████████████
  elo_home            2188  ████████████████████████
  wr_home             1450  ████████████████
  wr_away             1319  ██████████████
  is_knockout          257  ██

What this means:
  wr_diff  — Historical win rate gap between teams is the strongest signal
  elo_diff — Elo gap is close second — captures recent performance trend
  elo_away — The away team's absolute Elo matters more than home team's
             (away teams must be truly strong to win at World Cups)
  is_knockout — Least important. The model struggles to use stage info.
               This is a weakness — draws are almost impossible in knockouts
               but the model still sometimes predicts them there.


In [12]:
# ── Feature Importance visualization ──────────────────────────────────────
fi_df = pd.DataFrame(fi.items(), columns=['feature','importance']).sort_values('importance')

fig = px.bar(
    fi_df,
    x='importance', y='feature',
    orientation='h',
    color='importance',
    color_continuous_scale='Blues',
    title='LightGBM Feature Importance — What drives the predictions?',
    template='plotly_white'
)
fig.update_layout(coloraxis_showscale=False, yaxis={'categoryorder':'total ascending'})
fig.show()

## 6. Model Comparison

In [13]:
# ── Side-by-side comparison ────────────────────────────────────────────────
comparison = pd.DataFrame([
    {'model': 'FIFA Ranking (baseline)', 'accuracy': rank_acc, 'weighted_f1': rank_f1, 'log_loss': None},
    {'model': 'Logistic Regression',     'accuracy': lr_acc,   'weighted_f1': lr_f1,   'log_loss': None},
    {'model': 'LightGBM',                'accuracy': lgb_acc,  'weighted_f1': lgb_f1,  'log_loss': lgb_ll},
])

print('=== Model Comparison (Qatar 2022 test set) ===')
print(comparison.round(3).to_string(index=False))
print()
print('💡 Honest interpretation:')
print('   - FIFA ranking has the highest raw accuracy')
print('   - But it rarely predicts draws (inflating accuracy)')
print('   - LightGBM has the best F1 — more balanced across all 3 classes')
print('   - The gap is small because 64 matches is a very small test set')
print('   - Real value: LightGBM gives PROBABILITIES, ranking gives yes/no')

=== Model Comparison (Qatar 2022 test set) ===
                  model  accuracy  weighted_f1  log_loss
FIFA Ranking (baseline)     0.517        0.517       NaN
    Logistic Regression     0.469        0.394       NaN
               LightGBM     0.484        0.478     1.204

💡 Honest interpretation:
   - FIFA ranking has the highest raw accuracy
   - But it rarely predicts draws (inflating accuracy)
   - LightGBM has the best F1 — more balanced across all 3 classes
   - The gap is small because 64 matches is a very small test set
   - Real value: LightGBM gives PROBABILITIES, ranking gives yes/no


## 7. Error Analysis — The Morocco Story

This is the most important section. **Understanding why the model fails** tells us more than knowing it succeeds.

Morocco reached the semi-finals of Qatar 2022. The model didn't predict that.
Why? And what does it teach us about the architecture?

In [14]:
# ── What did the model get wrong? ─────────────────────────────────────────
test_analysis = test.copy().reset_index(drop=True)
test_analysis['lgb_pred']      = lgb_pred
test_analysis['lgb_pred_name'] = [REV_MAP[p] for p in lgb_pred]
test_analysis['actual_name']   = [REV_MAP[t] for t in y_test]
test_analysis['correct']       = test_analysis['lgb_pred'] == test_analysis['target']

# Add probabilities
test_analysis['p_home_win'] = lgb_proba[:, 0]
test_analysis['p_draw']     = lgb_proba[:, 1]
test_analysis['p_away_win'] = lgb_proba[:, 2]

# Misses
misses = test_analysis[~test_analysis['correct']].copy()
print(f'Correct predictions: {test_analysis["correct"].sum()} / {len(test_analysis)}')
print(f'Missed predictions:  {(~test_analysis["correct"]).sum()}')
print()

# Morocco specifically
morocco = test_analysis[
    (test_analysis['home_team'] == 'Morocco') |
    (test_analysis['away_team'] == 'Morocco')
].copy()

print('=== Morocco matches — model vs reality ===')
cols = ['home_team','away_team','actual_name','lgb_pred_name','p_home_win','p_draw','p_away_win','correct']
print(morocco[cols].round(3).to_string(index=False))

Correct predictions: 31 / 64
Missed predictions:  33

=== Morocco matches — model vs reality ===
home_team away_team actual_name lgb_pred_name  p_home_win  p_draw  p_away_win  correct
  Croatia   Morocco    home_win      home_win       0.622   0.356       0.022     True
   France   Morocco    home_win      home_win       0.519   0.466       0.015     True
  Morocco  Portugal    home_win      away_win       0.063   0.084       0.852    False
  Morocco     Spain        draw      away_win       0.029   0.121       0.851    False
   Canada   Morocco    away_win      away_win       0.263   0.364       0.373     True
  Belgium   Morocco    away_win          draw       0.081   0.844       0.075    False
  Morocco   Croatia        draw      away_win       0.029   0.396       0.575    False


In [15]:
# ── Why did the model miss Morocco? ───────────────────────────────────────
# Let's look at Morocco's Elo and win rate entering Qatar 2022

morocco_elo = elo_entry_df[
    (elo_entry_df['team'] == 'Morocco') &
    (elo_entry_df['year'] == 2022)
]
morocco_wr = win_rate.get('Morocco', 0.33)

# Compare to the teams they upset
comparison_teams = ['Morocco', 'Spain', 'Portugal', 'France', 'Belgium', 'Brazil', 'Argentina']
elo_compare = elo_entry_df[
    (elo_entry_df['team'].isin(comparison_teams)) &
    (elo_entry_df['year'] == 2022)
].copy()
elo_compare['win_rate'] = elo_compare['team'].map(win_rate)
elo_compare = elo_compare.sort_values('elo_entry', ascending=False)

print('Elo + Win Rate entering Qatar 2022:')
print(elo_compare[['team','elo_entry','win_rate']].round(3).to_string(index=False))

print()
print('=== The core problem ===')
print(f'Morocco Elo entering 2022:    {morocco_elo["elo_entry"].values[0]:.0f}')
print(f'Morocco historical win rate:  {morocco_wr:.1%}')
print()
print('The model sees a low-Elo, low-win-rate team.')
print('It correctly learned from 92 years of history that such teams usually lose.')
print('But Morocco 2022 was not the Morocco of 92 years of history.')
print()
print('What the model could NOT see:')
print('  → Their defensive organization under Regragui (new coach since 2022)')
print('  → Elite defensive PIS: top 3 in the tournament on tackles + interceptions')
print('  → A squad peaking at exactly the right moment')
print()
print('This is exactly why the Player Impact Score layer exists in the architecture.')
print('Historical Elo looks backward. PIS looks at who is actually on the pitch.')

Elo + Win Rate entering Qatar 2022:
     team  elo_entry  win_rate
   Brazil   1707.271     0.667
   France   1678.838     0.534
Argentina   1637.150     0.534
    Spain   1605.481     0.463
  Belgium   1605.317     0.412
 Portugal   1552.855     0.486
  Morocco   1435.006     0.217

=== The core problem ===
Morocco Elo entering 2022:    1435
Morocco historical win rate:  21.7%

The model sees a low-Elo, low-win-rate team.
It correctly learned from 92 years of history that such teams usually lose.
But Morocco 2022 was not the Morocco of 92 years of history.

What the model could NOT see:
  → Their defensive organization under Regragui (new coach since 2022)
  → Elite defensive PIS: top 3 in the tournament on tackles + interceptions
  → A squad peaking at exactly the right moment

This is exactly why the Player Impact Score layer exists in the architecture.
Historical Elo looks backward. PIS looks at who is actually on the pitch.


In [16]:
# ── Full miss analysis — all notable upsets ────────────────────────────────
print('All misses — Qatar 2022 (what the model got wrong):')
print()
miss_cols = ['home_team','away_team','actual_name','lgb_pred_name','p_home_win','p_draw','p_away_win']
print(misses[miss_cols].round(3).to_string(index=False))
print()

# Pattern in misses
print('Pattern analysis:')
print('  Missed draws:', len(misses[misses['actual_name']=='draw']))
print('  Missed upsets (away wins predicted as home wins):',
      len(misses[(misses['actual_name']=='away_win') & (misses['lgb_pred_name']=='home_win')]))
print()
print('💡 The model\'s weaknesses:')
print('   1. Draws are systematically under-predicted')
print('      → Solution: add draw-specific features (e.g. teams with similar strength)')
print('   2. Upsets by historically weak teams are missed')
print('      → Solution: add current squad quality signal (PIS)')
print('   3. Small test set (64 matches) — estimates have high variance')
print('      → Solution: cross-validate across multiple tournaments')

All misses — Qatar 2022 (what the model got wrong):

     home_team     away_team actual_name lgb_pred_name  p_home_win  p_draw  p_away_win
     Argentina        France        draw      away_win       0.347   0.180       0.474
       Morocco      Portugal    home_win      away_win       0.063   0.084       0.852
       Croatia        Brazil        draw      away_win       0.058   0.011       0.931
       Morocco         Spain        draw      away_win       0.029   0.121       0.851
         Japan       Croatia        draw      away_win       0.061   0.291       0.648
        France        Poland    home_win      away_win       0.159   0.091       0.751
Korea Republic      Portugal    home_win      away_win       0.170   0.148       0.681
         Ghana       Uruguay    away_win          draw       0.292   0.415       0.293
      Cameroon        Brazil    home_win      away_win       0.125   0.034       0.842
        Serbia   Switzerland    away_win      home_win       0.542   0.159   

In [17]:
# ── Confusion matrix ──────────────────────────────────────────────────────
cm = confusion_matrix(y_test, lgb_pred)
cm_df = pd.DataFrame(
    cm,
    index=['Actual: home_win','Actual: draw','Actual: away_win'],
    columns=['Pred: home_win','Pred: draw','Pred: away_win']
)

fig = px.imshow(
    cm_df,
    text_auto=True,
    color_continuous_scale='Blues',
    title='Confusion Matrix — LightGBM on Qatar 2022 (64 matches)',
    template='plotly_white'
)
fig.show()
print('Read: rows = what actually happened, columns = what the model predicted')
print('Diagonal = correct predictions. Off-diagonal = mistakes.')
print(f'The draw row shows the model missed {cm[1][0]+cm[1][2]} of {cm[1].sum()} draws.')

Read: rows = what actually happened, columns = what the model predicted
Diagonal = correct predictions. Off-diagonal = mistakes.
The draw row shows the model missed 11 of 15 draws.


## 8. Build WC 2026 Team Features

Now we apply the model forward — to predict World Cup 2026.

**The challenge:** We don't have 2026 player stats yet. The tournament hasn't happened.

**Our approach:**
- Use final Elo ratings (post Qatar 2022) as baseline team strength
- Use historical win rates as long-term quality signal
- Acknowledge uncertainty — we're extrapolating, not predicting with certainty

In [18]:
# ── WC 2026 qualified teams ────────────────────────────────────────────────
# 48 teams qualify for 2026 (expanded format)
# Using confirmed qualifiers + likely qualifiers based on recent performance
# Source: FIFA.com qualification results as of early 2026

WC2026_TEAMS = [
    # CONMEBOL (6 + host)
    'Argentina', 'Brazil', 'Uruguay', 'Colombia', 'Ecuador', 'Venezuela',
    # UEFA (16)
    'Spain', 'England', 'France', 'Germany', 'Portugal', 'Netherlands',
    'Belgium', 'Croatia', 'Serbia', 'Austria', 'Switzerland', 'Denmark',
    'Poland', 'Scotland', 'Czech Republic', 'Slovakia',
    # CONCACAF (6 + 3 hosts)
    'United States', 'Mexico', 'Canada', 'Panama', 'Honduras', 'Costa Rica',
    # CAF (9)
    'Morocco', 'Senegal', 'Nigeria', 'Egypt', 'Cameroon',
    'South Africa', 'Tunisia', 'Ghana', 'DR Congo',
    # AFC (8)
    'Japan', 'Korea Republic', 'Iran', 'Saudi Arabia',
    'Australia', 'Qatar', 'Jordan', 'Iraq',
    # OFC (1)
    'New Zealand',
    # Playoffs
    'Turkey', 'Ukraine', 'Romania',
]

# Build feature vector for each team
team_features_2026 = []
for team in WC2026_TEAMS:
    elo  = elo_final.get(team, 1500)            # final Elo post Qatar 2022
    wr   = win_rate.get(team, 0.25)             # historical win rate
    rank = rank_map.get(team, 100)              # FIFA ranking (Oct 2022)
    team_features_2026.append({
        'team': team, 'elo': elo, 'win_rate': wr, 'fifa_rank': rank
    })

teams_2026 = pd.DataFrame(team_features_2026).sort_values('elo', ascending=False)
print('WC 2026 teams by Elo strength (top 20):')
print(teams_2026.head(20).round(2).to_string(index=False))

WC 2026 teams by Elo strength (top 20):
       team      elo  win_rate  fifa_rank
Netherlands 1704.440     0.550          8
     Brazil 1692.770     0.670          1
     France 1692.650     0.530          4
    Germany 1667.370     0.660         11
  Argentina 1655.620     0.530          3
      Spain 1591.770     0.460          7
    Belgium 1590.030     0.410          2
    England 1582.900     0.430          5
    Croatia 1575.700     0.430         12
   Portugal 1557.480     0.490          9
    Uruguay 1548.450     0.420         14
    Romania 1526.980     0.380         53
    Senegal 1526.110     0.420         18
   Colombia 1525.780     0.410         17
     Mexico 1524.580     0.280         13
    Ecuador 1515.970     0.380         44
    Denmark 1514.830     0.390         10
     Poland 1508.360     0.450         26
Switzerland 1503.110     0.340         15
    Ukraine 1501.690     0.400         27


## 9. Group Stage Predictions

WC 2026 has 12 groups of 4 teams each (48 teams total, expanded format).
We predict every match in the group stage and compute expected points.

In [19]:
# ── Prediction function ────────────────────────────────────────────────────
def predict_match(home_team, away_team, stage='group', verbose=True):
    """
    Predict a single match using the LightGBM model.
    Returns: probabilities for [home_win, draw, away_win]
    """
    elo_h = elo_final.get(home_team, 1500)
    elo_a = elo_final.get(away_team, 1500)
    wr_h  = win_rate.get(home_team, 0.25)
    wr_a  = win_rate.get(away_team, 0.25)
    is_ko = 1 if stage == 'knockout' else 0

    features = pd.DataFrame([{
        'elo_diff': elo_h - elo_a,
        'elo_home': elo_h,
        'elo_away': elo_a,
        'wr_diff':  wr_h - wr_a,
        'wr_home':  wr_h,
        'wr_away':  wr_a,
        'is_knockout': is_ko
    }])

    proba = lgbm.predict_proba(features)[0]
    pred  = REV_MAP[lgbm.predict(features)[0]]

    if verbose:
        print(f'{home_team:20} vs {away_team:20}')
        print(f'  Home win: {proba[0]:.1%}  |  Draw: {proba[1]:.1%}  |  Away win: {proba[2]:.1%}')
        print(f'  Prediction: {pred}')
        print()

    return {'home_win': proba[0], 'draw': proba[1], 'away_win': proba[2], 'prediction': pred}

# Test a few matches
print('=== Sample predictions ===')
_ = predict_match('Argentina', 'Morocco')
_ = predict_match('England', 'France')
_ = predict_match('Morocco', 'Spain')

=== Sample predictions ===
Argentina            vs Morocco             
  Home win: 47.4%  |  Draw: 23.1%  |  Away win: 29.5%
  Prediction: home_win

England              vs France              
  Home win: 38.3%  |  Draw: 12.7%  |  Away win: 49.0%
  Prediction: away_win

Morocco              vs Spain               
  Home win: 8.8%  |  Draw: 19.3%  |  Away win: 71.9%
  Prediction: away_win



In [20]:
# ── Simulate group stage ───────────────────────────────────────────────────
# WC 2026 groups (preliminary draw — adjust when official draw is confirmed)
# This is a representative example — update with real groups after the draw

GROUPS = {
    'A': ['United States', 'Mexico', 'Canada', 'Uruguay'],
    'B': ['Argentina', 'Brazil', 'Colombia', 'Ecuador'],
    'C': ['France', 'Germany', 'Portugal', 'Morocco'],
    'D': ['Spain', 'England', 'Netherlands', 'Croatia'],
    'E': ['Japan', 'Korea Republic', 'Saudi Arabia', 'Senegal'],
    'F': ['Belgium', 'Serbia', 'Denmark', 'Poland'],
}

def simulate_group(group_name, teams):
    """Simulate all matches in a group and compute expected points."""
    from itertools import combinations

    # Expected points: win=3, draw=1, loss=0, weighted by probability
    exp_points = {team: 0.0 for team in teams}
    matches_played = []

    for home, away in combinations(teams, 2):
        result = predict_match(home, away, stage='group', verbose=False)
        # Expected points for each team
        exp_points[home] += result['home_win'] * 3 + result['draw'] * 1
        exp_points[away] += result['away_win'] * 3 + result['draw'] * 1
        matches_played.append({
            'home': home, 'away': away,
            'p_home': result['home_win'],
            'p_draw': result['draw'],
            'p_away': result['away_win'],
            'predicted': result['prediction']
        })

    # Standings
    standings = pd.DataFrame([
        {'team': t, 'exp_points': p, 'elo': elo_final.get(t, 1500)}
        for t, p in exp_points.items()
    ]).sort_values('exp_points', ascending=False)
    standings['position'] = range(1, len(standings)+1)

    return standings, matches_played

# Run all groups
all_group_results = {}
print('='*60)
print('GROUP STAGE PREDICTIONS — World Cup 2026')
print('='*60)
for group_name, teams in GROUPS.items():
    standings, matches = simulate_group(group_name, teams)
    all_group_results[group_name] = {'standings': standings, 'matches': matches}
    print(f'\nGroup {group_name}:')
    print(standings[['position','team','exp_points','elo']].round(2).to_string(index=False))
    print(f'→ Predicted qualifiers: {standings.iloc[0]["team"]} and {standings.iloc[1]["team"]}')

GROUP STAGE PREDICTIONS — World Cup 2026

Group A:
 position          team  exp_points      elo
        1       Uruguay       5.890 1548.450
        2        Mexico       4.410 1524.580
        3 United States       3.410 1456.050
        4        Canada       2.240 1421.290
→ Predicted qualifiers: Uruguay and Mexico

Group B:
 position      team  exp_points      elo
        1    Brazil       6.340 1692.770
        2 Argentina       4.850 1655.620
        3   Ecuador       2.770 1515.970
        4  Colombia       2.750 1525.780
→ Predicted qualifiers: Brazil and Argentina

Group C:
 position     team  exp_points      elo
        1  Germany       7.780 1667.370
        2 Portugal       4.660 1557.480
        3   France       2.370 1692.650
        4  Morocco       1.790 1485.070
→ Predicted qualifiers: Germany and Portugal

Group D:
 position        team  exp_points      elo
        1 Netherlands       5.380 1704.440
        2       Spain       3.910 1591.770
        3     England      

In [21]:
# ── Visualize group results ────────────────────────────────────────────────
all_standings = []
for group_name, results in all_group_results.items():
    df = results['standings'].copy()
    df['group'] = f'Group {group_name}'
    all_standings.append(df)

standings_all = pd.concat(all_standings)

fig = px.bar(
    standings_all,
    x='exp_points', y='team',
    color='group',
    facet_col='group',
    facet_col_wrap=3,
    orientation='h',
    title='Expected Points by Group — World Cup 2026 AI Predictions',
    labels={'exp_points': 'Expected Points', 'team': ''},
    template='plotly_white',
    height=700
)
fig.update_layout(showlegend=False)
fig.show()

print('💡 Expected points = sum of (probability of win × 3 + probability of draw × 1)')
print('   across all group matches. More honest than binary win/loss predictions.')

💡 Expected points = sum of (probability of win × 3 + probability of draw × 1)
   across all group matches. More honest than binary win/loss predictions.


In [22]:
# ── Save all predictions ───────────────────────────────────────────────────
standings_all.to_csv(DATA_PATH + 'wc2026_group_predictions.csv', index=False)
teams_2026.to_csv(DATA_PATH + 'wc2026_team_strength.csv', index=False)

print('✅ Saved: wc2026_group_predictions.csv')
print('✅ Saved: wc2026_team_strength.csv')
print()
print('🎉 Day 2 complete!')
print()
print('What you built:')
print('  ✅ FIFA ranking baseline — your floor to beat')
print('  ✅ Logistic Regression — linear model with Elo features')
print('  ✅ LightGBM — main predictive model')
print('  ✅ Leave-one-tournament-out validation on Qatar 2022')
print('  ✅ Error analysis — understood WHY Morocco was missed')
print('  ✅ Group stage predictions for WC 2026 with probabilities')
print()
print('Day 3 plan: Streamlit app + LLM narrative layer (LangChain + Gemini)')

✅ Saved: wc2026_group_predictions.csv
✅ Saved: wc2026_team_strength.csv

🎉 Day 2 complete!

What you built:
  ✅ FIFA ranking baseline — your floor to beat
  ✅ Logistic Regression — linear model with Elo features
  ✅ LightGBM — main predictive model
  ✅ Leave-one-tournament-out validation on Qatar 2022
  ✅ Error analysis — understood WHY Morocco was missed
  ✅ Group stage predictions for WC 2026 with probabilities

Day 3 plan: Streamlit app + LLM narrative layer (LangChain + Gemini)


---

## ✅ Day 2 Summary

### Key learnings

**On model selection:**  
LightGBM wins on F1 score, not raw accuracy. Always choose metrics that match your actual problem — in multi-class classification with imbalanced classes, accuracy misleads.

**On feature importance:**  
Historical win rate differential is the strongest predictor — stronger even than Elo. This makes intuitive sense: teams that consistently win World Cup matches have something that goes beyond any single tournament's form.

**On error analysis:**  
Morocco is not a model failure — it's a lesson in the limits of backward-looking features. The fix is already built: the Player Impact Score captures current squad quality that Elo can't see.

**On uncertainty:**  
Probabilistic predictions (48% home win, 28% draw, 24% away win) are more valuable than binary predictions. They quantify what we don't know, which is honest and useful.

### Architecture so far

```
Historical data (1930–2022)
        ↓
Elo Rating System  +  Historical Win Rates
        ↓
LightGBM Classifier  →  Match probabilities
        ↓
Player Impact Score  →  Squad quality adjustment  [Day 1]
        ↓
LLM Narrative Layer  →  Human-readable analysis   [Day 3]
        ↓
Streamlit App        →  Public deployment          [Day 3]
```

### Day 3 plan
1. Build the LLM narrative layer with LangChain + Gemini
2. Build the Streamlit interface
3. Deploy on Hugging Face Spaces
4. Generate match analysis for every group stage game